# AMD-Latent
### Evaluation AMD classifiers

---

The current notebook evaluates and compares the different performances of classifiers trained to distinguish between AMD score classes

# Packages import and functions defining

Loading the required packages and definition of functions.

In [ ]:
# PACKAGES IMPORT

# System packages
import os
import numpy as np
from pathlib import Path
import pandas as pd
import pickle as pkl
from tqdm import tqdm

# Jupyter packages
import ipywidgets as widgets
from amdlatent.utils.widgets import toggleoptions, filechooser, slideroptions

# Graphing packages
from amdlatent.models.evaluation import metricsgraph, robustnesstable, statisticaldiff, weightedbalancedaccuracy

# Classification packages
from amdlatent.dataset.extraction import setsextraction
from amdlatent.dataset.audit import patientbootstrapindex, bootstrapci, bhfdr

# Path checker
if Path().resolve().name != "AMD-Latent": os.chdir(os.path.join(Path().resolve().parents[1]))

# Code execution

Select the following options for running the script and execute it.

In [ ]:
# OPTIONS SELECTION

clipselect = toggleoptions(['RETCLIP', 'CLIP', 'ResNet50'], "Select the <b>model</b> to use:")

classselect = toggleoptions(['All', 'Binary No', 'Binary Late'], "Select the <b>classification type</b> to use:")
splitselect = toggleoptions(["Couples-wise", "Patient-wise", "Couples stratified-wise"],"Select the <b>dataset split type</b>:")
setselect = toggleoptions(['Training', 'Validation', 'Test'], "Select the <b>subset</b> on which visualize graphs:")
infoselect = toggleoptions(['Bars with STD', 'Bars with Min-Max', 'Robustness', 'Robustness table', 'Robustness statistical', 'Robustness bootstrap'], "Select the <b>type of info</b> to visualize:")

graphbox = widgets.Output()
display(graphbox)
graphselect = None

def _update_graphselect(change=None):
    global graphselect
    graphbox.clear_output()
    if infoselect.value == "Bars with STD" or infoselect.value == "Bars with Min-Max" or infoselect.value == "Robustness" or infoselect.value == "Robustness statistical":
        with graphbox:
            graphselect = toggleoptions(['Balanced Accuracy', 'Accuracy', 'MAE', 'Class Accuracy'], "Select the <b>graph</b> to display:")
    else: 
        graphselect = None

def _bind_graphselect(change=None):
    if infoselect is not None:
        infoselect.observe(_update_graphselect, names="value")
    _update_graphselect()
_bind_graphselect()

bootbox = widgets.Output()
display(bootbox)
dirselect = None
bootreplicatesselect = None

def _update_bootbox(change=None):
    global dirselect, bootreplicatesselect
    bootbox.clear_output()
    if infoselect.value == "Robustness bootstrap":
        with bootbox:
            dirselect = filechooser("Select the <b>AMD Study</b> folder:")
            bootreplicatesselect = slideroptions("Select the <b>number of patient-cluster bootstrap replicates</b>:", 100, 2000, 100, 2000)
    else:
        dirselect = None
        bootreplicatesselect = None

def _bind_bootbox(change=None):
    if infoselect is not None:
        infoselect.observe(_update_bootbox, names="value")
    _update_bootbox()
_bind_bootbox()

In [ ]:
# SCRIPT EXECUTION

metadata = {
    "clip": clipselect.value if clipselect is not None else None,
    "classtype": classselect.value if classselect is not None else None,
    "splittype": splitselect.value if splitselect is not None else None,
    "subset": setselect.value if setselect is not None else None,
    "graph": graphselect.value if graphselect is not None else None,
    "info": infoselect.value if infoselect is not None else None,
    "directory": dirselect.selected_path if dirselect is not None else None,
    "bootreplicates": bootreplicatesselect.value if bootreplicatesselect is not None else None,
}

if metadata["info"] == "Robustness table":
    display(robustnesstable(metadata["classtype"], metadata["subset"], metadata["splittype"], metadata["clip"]))
elif metadata["info"] == "Robustness statistical":
    display(statisticaldiff(metadata["classtype"], metadata["graph"], metadata["subset"], splittype=metadata["splittype"], encoder=metadata["clip"]))
elif metadata["info"] == "Robustness bootstrap":
    dataset = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='AMD')
    dataset.columns = dataset.columns.astype(str).str.strip()
    classifiers = ['Random Forest', 'AdaBoost', 'SVM', 'SVM Linear', 'MLP']
    _, _, _, _, X_test, y_test, _, testpaths = setsextraction(dataset, metadata["classtype"], metadata["splittype"], testpath=True, encoder=metadata["clip"].lower())
    patientids = np.array([str(Path(p[0]).parts[-5]) for p in testpaths])
    predictionsdir = os.path.join("results", "areds", "classifiers", metadata["clip"].lower(), "predictions", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''))
    preds = {}
    for clf in classifiers:
        predpath = os.path.join(predictionsdir, f"{clf.lower().replace(' ', '_')}.pkl")
        if not os.path.exists(predpath): raise FileNotFoundError()
        with open(predpath, 'rb') as f: cached = pkl.load(f)
        if not np.array_equal(np.asarray(cached["test_true"]), np.asarray(y_test)):
            raise ValueError(f"Cached predictions for {clf} ({predpath}) do not match the current test set")
        preds[clf] = cached["test_preds"]

    n_boot = int(metadata["bootreplicates"])
    rng = np.random.default_rng(42)
    pairs = [(a, b) for i, a in enumerate(classifiers) for b in classifiers[i + 1:]]
    accs = {clf: np.full(n_boot, np.nan) for clf in classifiers}
    diffs = {p: np.full(n_boot, np.nan) for p in pairs}
    for b in tqdm(range(n_boot), desc="Patient-cluster bootstrap"):
        idx = patientbootstrapindex(patientids, rng)
        weights = np.bincount(idx, minlength=len(y_test)).astype(float)
        repaccs = {clf: float(np.mean(weightedbalancedaccuracy(y_test, preds[clf], weights))) for clf in classifiers}
        for clf in classifiers:
            accs[clf][b] = repaccs[clf]
        for a, c in pairs:
            diffs[(a, c)][b] = repaccs[a] - repaccs[c]

    accrows = []
    for clf in classifiers:
        low, high, _, nok = bootstrapci(accs[clf])
        accrows.append({
            "Classifier": clf,
            "Repetitions (cached)": preds[clf].shape[0],
            "Balanced accuracy (mean over repetitions)": float(np.nanmean(accs[clf])),
            "95% CI low": low,
            "95% CI high": high,
            "Valid replicates": nok,
        })
    accdf = pd.DataFrame(accrows)
    print(f"Patient-cluster bootstrap balanced accuracy (cached predictions from {predictionsdir}, {n_boot} bootstrap replicates):")
    display(accdf)

    pairrows = []
    pvals = []
    for a, c in pairs:
        low, high, pval, nok = bootstrapci(diffs[(a, c)])
        pvals.append(pval)
        pairrows.append({
            "Pair (A - B)": f"{a} - {c}",
            "Observed difference": float(np.nanmean(diffs[(a, c)])),
            "95% CI low": low,
            "95% CI high": high,
            "Bootstrap p-value": pval,
            "CI excludes 0": bool(low > 0 or high < 0),
            "Valid replicates": nok,
        })
    pairdf = pd.DataFrame(pairrows)
    reject, p_adj = bhfdr(pvals)
    pairdf["p-value (BH-FDR)"] = p_adj
    pairdf["Significant (BH 0.05)"] = reject
    print("\nPairwise classifier comparison (same bootstrap draw for both classifiers in each pair, so the correlation between their errors on shared patients is preserved):")
    display(pairdf)
else:
    metricsgraph(metadata["classtype"], metadata["graph"], metadata["subset"], metadata["info"], splittype=metadata["splittype"], encoder=metadata["clip"])